## 📚 Import Libraries


In [ ]:
# Data processing
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("seaborn-v0_8-darkgrid")

# ML models
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    StackingClassifier,
    VotingClassifier,
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# Preprocessing & validation
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    RandomizedSearchCV,
    StratifiedKFold,
    cross_val_score,
)
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_selection import SelectFromModel, RFE
from imblearn.over_sampling import SMOTE

# Metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

# Save models
import joblib
import json
from pathlib import Path

print("✅ Libraries imported successfully!")

✅ Libraries imported successfully!


## 📂 Load Dataset


In [ ]:
# Load data
df = pd.read_csv("datasets/dataset.csv")

print(f"Dataset shape: {df.shape}")
print(f"\nTarget distribution:")
print(df["Target"].value_counts())
print(f"\nPercentages:")
print(df["Target"].value_counts(normalize=True) * 100)

Dataset shape: (4424, 35)

Target distribution:
Target
Graduate    2209
Dropout     1421
Enrolled     794
Name: count, dtype: int64

Percentages:
Target
Graduate    49.932188
Dropout     32.120253
Enrolled    17.947559
Name: proportion, dtype: float64


## 🔧 Feature Engineering - Create New Features


In [ ]:
print("🔧 Creating engineered features...\n")

# 1. Academic Performance Metrics
df["avg_approved"] = (
    df["Curricular units 1st sem (approved)"]
    + df["Curricular units 2nd sem (approved)"]
) / 2

df["avg_grade"] = (
    df["Curricular units 1st sem (grade)"] + df["Curricular units 2nd sem (grade)"]
) / 2

df["grade_consistency"] = abs(
    df["Curricular units 1st sem (grade)"] - df["Curricular units 2nd sem (grade)"]
)

df["grade_improvement"] = (
    df["Curricular units 2nd sem (grade)"] - df["Curricular units 1st sem (grade)"]
)

# 2. Failure Rates
df["failure_rate_sem1"] = np.where(
    df["Curricular units 1st sem (evaluations)"] > 0,
    (
        df["Curricular units 1st sem (evaluations)"]
        - df["Curricular units 1st sem (approved)"]
    )
    / df["Curricular units 1st sem (evaluations)"],
    0,
)

df["failure_rate_sem2"] = np.where(
    df["Curricular units 2nd sem (evaluations)"] > 0,
    (
        df["Curricular units 2nd sem (evaluations)"]
        - df["Curricular units 2nd sem (approved)"]
    )
    / df["Curricular units 2nd sem (evaluations)"],
    0,
)

df["total_failure_rate"] = (df["failure_rate_sem1"] + df["failure_rate_sem2"]) / 2

# 3. Course Load & Completion
df["total_evaluations"] = (
    df["Curricular units 1st sem (evaluations)"]
    + df["Curricular units 2nd sem (evaluations)"]
)

df["total_approved"] = (
    df["Curricular units 1st sem (approved)"]
    + df["Curricular units 2nd sem (approved)"]
)

df["completion_rate"] = np.where(
    df["total_evaluations"] > 0, df["total_approved"] / df["total_evaluations"], 0
)

# 4. Financial Stability Score
df["financial_stability"] = (
    df["Tuition fees up to date"] + df["Scholarship holder"] - df["Debtor"]
).clip(0, 3)

# 5. Family Education Level
df["parent_education_avg"] = (
    df["Mother's qualification"] + df["Father's qualification"]
) / 2

df["parent_education_max"] = df[
    ["Mother's qualification", "Father's qualification"]
].max(axis=1)

# 6. Age Categories
df["is_mature_student"] = (df["Age at enrollment"] >= 25).astype(int)
df["is_traditional_age"] = (
    (df["Age at enrollment"] >= 18) & (df["Age at enrollment"] <= 22)
).astype(int)

# 7. Academic Risk Indicators
df["has_sem1_failures"] = (
    df["Curricular units 1st sem (approved)"]
    < df["Curricular units 1st sem (evaluations)"]
).astype(int)

df["has_sem2_failures"] = (
    df["Curricular units 2nd sem (approved)"]
    < df["Curricular units 2nd sem (evaluations)"]
).astype(int)

df["both_sems_failures"] = (df["has_sem1_failures"] & df["has_sem2_failures"]).astype(
    int
)

# 8. Grade Performance Categories
df["high_performer"] = (df["avg_grade"] >= 14).astype(int)
df["low_performer"] = (df["avg_grade"] < 10).astype(int)

print("✅ Created 23 new engineered features:")
engineered_features = [
    "avg_approved",
    "avg_grade",
    "grade_consistency",
    "grade_improvement",
    "failure_rate_sem1",
    "failure_rate_sem2",
    "total_failure_rate",
    "total_evaluations",
    "total_approved",
    "completion_rate",
    "financial_stability",
    "parent_education_avg",
    "parent_education_max",
    "is_mature_student",
    "is_traditional_age",
    "has_sem1_failures",
    "has_sem2_failures",
    "both_sems_failures",
    "high_performer",
    "low_performer",
]
for i, feat in enumerate(engineered_features, 1):
    print(f"  {i:2d}. {feat}")

print(f"\nNew dataset shape: {df.shape}")

🔧 Creating engineered features...

✅ Created 23 new engineered features:
   1. avg_approved
   2. avg_grade
   3. grade_consistency
   4. grade_improvement
   5. failure_rate_sem1
   6. failure_rate_sem2
   7. total_failure_rate
   8. total_evaluations
   9. total_approved
  10. completion_rate
  11. financial_stability
  12. parent_education_avg
  13. parent_education_max
  14. is_mature_student
  15. is_traditional_age
  16. has_sem1_failures
  17. has_sem2_failures
  18. both_sems_failures
  19. high_performer
  20. low_performer

New dataset shape: (4424, 55)


## 🎯 Feature Preparation


In [ ]:
# Drop features as planned (administrative/economic)
features_to_drop = [
    "Application mode",
    "Application order",
    "Course",
    "Daytime/evening attendance",
    "Unemployment rate",
    "Inflation rate",
    "GDP",
    "Target",  # Separate as y
]

# Prepare features and target
X = df.drop(columns=features_to_drop, errors="ignore")
y = df["Target"].copy()

# Encode target
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)

print(f"✅ Total features: {X.shape[1]}")
print(f"   - Original features: 28")
print(f"   - Engineered features: {len(engineered_features)}")
print(f"   - Total: {X.shape[1]}")
print(
    f"\nTarget encoding: {dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))}"
)

✅ Total features: 47
   - Original features: 28
   - Engineered features: 20
   - Total: 47

Target encoding: {'Dropout': np.int64(0), 'Enrolled': np.int64(1), 'Graduate': np.int64(2)}


## 🔀 Train-Test Split & Scaling


In [5]:
# Stratified split to preserve class distribution
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print(f"\nTraining class distribution:")
unique, counts = np.unique(y_train, return_counts=True)
for label, count in zip(label_encoder.inverse_transform(unique), counts):
    print(f"  {label}: {count} ({count/len(y_train)*100:.1f}%)")

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\n✅ Features scaled using StandardScaler")

Training set: 3539 samples
Test set: 885 samples

Training class distribution:
  Dropout: 1137 (32.1%)
  Enrolled: 635 (17.9%)
  Graduate: 1767 (49.9%)

✅ Features scaled using StandardScaler


## ⚖️ Handle Class Imbalance with SMOTE


In [6]:
print("⚖️ Applying SMOTE for class balancing...\n")

# Apply SMOTE
smote = SMOTE(random_state=42, k_neighbors=5)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_scaled, y_train)

print(f"Original training set: {X_train_scaled.shape[0]} samples")
print(f"Balanced training set: {X_train_balanced.shape[0]} samples")
print(f"\nBalanced class distribution:")
unique, counts = np.unique(y_train_balanced, return_counts=True)
for label, count in zip(label_encoder.inverse_transform(unique), counts):
    print(f"  {label}: {count} ({count/len(y_train_balanced)*100:.1f}%)")

print("\n✅ Classes balanced with SMOTE")

⚖️ Applying SMOTE for class balancing...

Original training set: 3539 samples
Balanced training set: 5301 samples

Balanced class distribution:
  Dropout: 1767 (33.3%)
  Enrolled: 1767 (33.3%)
  Graduate: 1767 (33.3%)

✅ Classes balanced with SMOTE


## 🌲 Baseline: Random Forest with Balanced Data


In [ ]:
print("🌲 Training Random Forest on balanced data...\n")

rf_baseline = RandomForestClassifier(
    n_estimators=300,
    max_depth=20,
    min_samples_split=10,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1,
)

rf_baseline.fit(X_train_balanced, y_train_balanced)
rf_pred = rf_baseline.predict(X_test_scaled)
rf_acc = accuracy_score(y_test, rf_pred)

print(f"✅ Random Forest Baseline (with SMOTE + Engineered Features)")
print(f"   Accuracy: {rf_acc*100:.2f}%")
print(f"\nClassification Report:")
print(classification_report(y_test, rf_pred, target_names=label_encoder.classes_))

# Feature importance
feature_importance = pd.DataFrame(
    {"feature": X.columns, "importance": rf_baseline.feature_importances_}
).sort_values("importance", ascending=False)

print("\nTop 15 Most Important Features:")
print(feature_importance.head(15).to_string(index=False))

🌲 Training Random Forest on balanced data...

✅ Random Forest Baseline (with SMOTE + Engineered Features)
   Accuracy: 76.61%

Classification Report:
              precision    recall  f1-score   support

     Dropout       0.84      0.72      0.77       284
    Enrolled       0.49      0.58      0.53       159
    Graduate       0.84      0.86      0.85       442

    accuracy                           0.77       885
   macro avg       0.72      0.72      0.72       885
weighted avg       0.78      0.77      0.77       885


Top 15 Most Important Features:
                            feature  importance
Curricular units 2nd sem (approved)    0.071779
                    completion_rate    0.064863
                     total_approved    0.062717
                       avg_approved    0.058970
                 total_failure_rate    0.049859
                          avg_grade    0.046919
   Curricular units 2nd sem (grade)    0.044359
                  failure_rate_sem2    0.043529
Curr

## 🚀 XGBoost - High Performance Gradient Boosting


In [ ]:
print("🚀 Training XGBoost...\n")

xgb_model = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    eval_metric="mlogloss",
)

xgb_model.fit(X_train_balanced, y_train_balanced)
xgb_pred = xgb_model.predict(X_test_scaled)
xgb_acc = accuracy_score(y_test, xgb_pred)

print(f"✅ XGBoost Accuracy: {xgb_acc*100:.2f}%")
print(f"\nClassification Report:")
print(classification_report(y_test, xgb_pred, target_names=label_encoder.classes_))

🚀 Training XGBoost...

✅ XGBoost Accuracy: 74.35%

Classification Report:
              precision    recall  f1-score   support

     Dropout       0.81      0.74      0.77       284
    Enrolled       0.43      0.44      0.43       159
    Graduate       0.82      0.86      0.84       442

    accuracy                           0.74       885
   macro avg       0.69      0.68      0.68       885
weighted avg       0.75      0.74      0.74       885



## 💡 LightGBM - Fast Gradient Boosting


In [ ]:
print("💡 Training LightGBM...\n")

lgbm_model = LGBMClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.1,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)

lgbm_model.fit(X_train_balanced, y_train_balanced)
lgbm_pred = lgbm_model.predict(X_test_scaled)
lgbm_acc = accuracy_score(y_test, lgbm_pred)

print(f"✅ LightGBM Accuracy: {lgbm_acc*100:.2f}%")
print(f"\nClassification Report:")
print(classification_report(y_test, lgbm_pred, target_names=label_encoder.classes_))

💡 Training LightGBM...

✅ LightGBM Accuracy: 74.80%

Classification Report:
              precision    recall  f1-score   support

     Dropout       0.79      0.75      0.77       284
    Enrolled       0.43      0.40      0.41       159
    Graduate       0.82      0.87      0.85       442

    accuracy                           0.75       885
   macro avg       0.68      0.67      0.68       885
weighted avg       0.74      0.75      0.74       885



## 🎯 Hyperparameter Tuning - XGBoost with RandomizedSearchCV


In [ ]:
print("🎯 Hyperparameter tuning XGBoost with RandomizedSearchCV...\n")

xgb_param_grid = {
    "n_estimators": [300, 500, 700, 1000],
    "max_depth": [4, 6, 8, 10],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "gamma": [0, 0.1, 0.2],
    "reg_alpha": [0, 0.1, 0.5],
    "reg_lambda": [0.5, 1.0, 2.0],
    "min_child_weight": [1, 3, 5],
}

xgb_random = RandomizedSearchCV(
    XGBClassifier(random_state=42, n_jobs=-1, eval_metric="mlogloss"),
    xgb_param_grid,
    n_iter=50,  # Try 50 random combinations
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    random_state=42,
    verbose=1,
)

import time

start_time = time.time()
xgb_random.fit(X_train_balanced, y_train_balanced)
tuning_time = time.time() - start_time

print(f"\n✅ Tuning completed in {tuning_time:.1f} seconds")
print(f"\nBest parameters: {xgb_random.best_params_}")
print(f"Best CV score: {xgb_random.best_score_*100:.2f}%")

xgb_tuned = xgb_random.best_estimator_
xgb_tuned_pred = xgb_tuned.predict(X_test_scaled)
xgb_tuned_acc = accuracy_score(y_test, xgb_tuned_pred)

print(f"\n✅ XGBoost Tuned Test Accuracy: {xgb_tuned_acc*100:.2f}%")
print(f"   Improvement: {(xgb_tuned_acc - xgb_acc)*100:+.2f}%")

🎯 Hyperparameter tuning XGBoost with RandomizedSearchCV...

Fitting 5 folds for each of 50 candidates, totalling 250 fits

✅ Tuning completed in 689.4 seconds

Best parameters: {'subsample': 0.8, 'reg_lambda': 1.0, 'reg_alpha': 0.5, 'n_estimators': 1000, 'min_child_weight': 1, 'max_depth': 10, 'learning_rate': 0.05, 'gamma': 0, 'colsample_bytree': 0.8}
Best CV score: 77.82%

✅ XGBoost Tuned Test Accuracy: 75.25%
   Improvement: +0.90%


## 🎯 Hyperparameter Tuning - Random Forest


In [ ]:
print("🎯 Hyperparameter tuning Random Forest with RandomizedSearchCV...\n")

rf_param_grid = {
    "n_estimators": [300, 500, 700, 1000],
    "max_depth": [15, 20, 25, 30, None],
    "min_samples_split": [2, 5, 10, 15],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2", 0.3, 0.5],
    "bootstrap": [True, False],
}

rf_random = RandomizedSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    rf_param_grid,
    n_iter=50,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    random_state=42,
    verbose=1,
)

start_time = time.time()
rf_random.fit(X_train_balanced, y_train_balanced)
tuning_time = time.time() - start_time

print(f"\n✅ Tuning completed in {tuning_time:.1f} seconds")
print(f"\nBest parameters: {rf_random.best_params_}")
print(f"Best CV score: {rf_random.best_score_*100:.2f}%")

rf_tuned = rf_random.best_estimator_
rf_tuned_pred = rf_tuned.predict(X_test_scaled)
rf_tuned_acc = accuracy_score(y_test, rf_tuned_pred)

print(f"\n✅ Random Forest Tuned Test Accuracy: {rf_tuned_acc*100:.2f}%")
print(f"   Improvement: {(rf_tuned_acc - rf_acc)*100:+.2f}%")

🎯 Hyperparameter tuning Random Forest with RandomizedSearchCV...

Fitting 5 folds for each of 50 candidates, totalling 250 fits

✅ Tuning completed in 912.7 seconds

Best parameters: {'n_estimators': 500, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': 25, 'bootstrap': False}
Best CV score: 84.27%

✅ Random Forest Tuned Test Accuracy: 75.37%
   Improvement: -1.24%


## 🏗️ Stacking Ensemble - Combine Best Models


In [ ]:
print("🏗️ Building Stacking Ensemble...\n")

# Base models
estimators = [("rf", rf_tuned), ("xgb", xgb_tuned), ("lgbm", lgbm_model)]

# Meta-learner
stacking_model = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(max_iter=1000),
    cv=5,
    n_jobs=-1,
)

stacking_model.fit(X_train_balanced, y_train_balanced)
stacking_pred = stacking_model.predict(X_test_scaled)
stacking_acc = accuracy_score(y_test, stacking_pred)

print(f"✅ Stacking Ensemble Accuracy: {stacking_acc*100:.2f}%")
print(f"\nClassification Report:")
print(classification_report(y_test, stacking_pred, target_names=label_encoder.classes_))

🏗️ Building Stacking Ensemble...

✅ Stacking Ensemble Accuracy: 74.92%

Classification Report:
              precision    recall  f1-score   support

     Dropout       0.79      0.77      0.78       284
    Enrolled       0.45      0.43      0.44       159
    Graduate       0.83      0.85      0.84       442

    accuracy                           0.75       885
   macro avg       0.69      0.68      0.69       885
weighted avg       0.75      0.75      0.75       885



## 🗳️ Voting Ensemble - Soft Voting


In [ ]:
print("🗳️ Building Voting Ensemble (Soft Voting)...\n")

voting_model = VotingClassifier(
    estimators=estimators, voting="soft", n_jobs=-1  # Use probability averaging
)

voting_model.fit(X_train_balanced, y_train_balanced)
voting_pred = voting_model.predict(X_test_scaled)
voting_acc = accuracy_score(y_test, voting_pred)

print(f"✅ Voting Ensemble Accuracy: {voting_acc*100:.2f}%")
print(f"\nClassification Report:")
print(classification_report(y_test, voting_pred, target_names=label_encoder.classes_))

🗳️ Building Voting Ensemble (Soft Voting)...



ValueError: The estimator XGBClassifier should be a classifier.

## 📊 Model Comparison - All Enhanced Models


In [ ]:
# Compile results
results = pd.DataFrame(
    {
        "Model": [
            "Stacking Ensemble",
            "Voting Ensemble",
            "XGBoost (Tuned)",
            "Random Forest (Tuned)",
            "LightGBM",
            "XGBoost (Baseline)",
            "Random Forest (Baseline)",
        ],
        "Accuracy": [
            stacking_acc,
            voting_acc,
            xgb_tuned_acc,
            rf_tuned_acc,
            lgbm_acc,
            xgb_acc,
            rf_acc,
        ],
        "Type": [
            "Ensemble",
            "Ensemble",
            "Gradient Boosting (Tuned)",
            "Random Forest (Tuned)",
            "Gradient Boosting",
            "Gradient Boosting",
            "Random Forest",
        ],
    }
).sort_values("Accuracy", ascending=False)

print("📊 Enhanced Model Performance Comparison:\n")
print(results.to_string(index=False))

# Visualization
plt.figure(figsize=(12, 6))
colors = [
    "#2ecc71" if "Ensemble" in t else "#3498db" if "Tuned" in t else "#95a5a6"
    for t in results["Type"]
]
bars = plt.barh(results["Model"], results["Accuracy"] * 100, color=colors)
plt.xlabel("Accuracy (%)", fontsize=12)
plt.title(
    "Enhanced Model Accuracy Comparison\n(Green=Ensemble, Blue=Tuned, Gray=Baseline)",
    fontsize=14,
    fontweight="bold",
)
plt.xlim(70, 85)
plt.axvline(
    x=75.14, color="red", linestyle="--", linewidth=2, label="Original RF 75.14%"
)

# Add value labels
for i, (bar, acc) in enumerate(zip(bars, results["Accuracy"])):
    plt.text(
        acc * 100 + 0.1,
        bar.get_y() + bar.get_height() / 2,
        f"{acc*100:.2f}%",
        va="center",
        fontsize=10,
        fontweight="bold",
    )

plt.legend()
plt.tight_layout()
plt.show()

print(f"\n🏆 Best Model: {results.iloc[0]['Model']}")
print(f"   Accuracy: {results.iloc[0]['Accuracy']*100:.2f}%")
print(
    f"   Improvement over original: {(results.iloc[0]['Accuracy'] - 0.7514)*100:+.2f}%"
)

NameError: name 'voting_acc' is not defined

## 🧪 Test Best Model with Diverse Student Profiles


In [ ]:
print("🧪 Testing Best Model with Diverse Student Profiles:\n")

# Select best model
best_model = stacking_model if stacking_acc >= voting_acc else voting_model
best_model_name = (
    "Stacking Ensemble" if stacking_acc >= voting_acc else "Voting Ensemble"
)

# Test with different student profiles from test set
test_indices = [50, 150, 250, 350, 450, 550]  # Sample diverse students

for idx in test_indices:
    student_features = X_test_scaled[idx : idx + 1]
    true_label = label_encoder.inverse_transform([y_test[idx]])[0]

    prediction = best_model.predict(student_features)[0]
    probabilities = best_model.predict_proba(student_features)[0]

    predicted_label = label_encoder.inverse_transform([prediction])[0]
    dropout_risk = probabilities[0] * 100

    # Get some key features for context
    original_idx = X_test.index[idx]
    age = df.loc[original_idx, "Age at enrollment"]
    sem2_approved = df.loc[original_idx, "Curricular units 2nd sem (approved)"]
    sem2_grade = df.loc[original_idx, "Curricular units 2nd sem (grade)"]
    tuition_paid = df.loc[original_idx, "Tuition fees up to date"]

    risk_level = (
        "🔴 HIGH"
        if dropout_risk >= 70
        else ("🟡 MODERATE" if dropout_risk >= 40 else "🟢 LOW")
    )

    print(f"Student Profile:")
    print(f"   Age: {age:.0f}")
    print(f"   2nd Sem Approved: {sem2_approved:.0f}")
    print(f"   2nd Sem Grade: {sem2_grade:.1f}")
    print(f"   Tuition Paid: {'Yes' if tuition_paid == 1 else 'No'}")
    print(f"   True Status: {true_label}")
    print(
        f"\n   {best_model_name}: {predicted_label} (Dropout risk: {dropout_risk:.1f}%)"
    )
    print(f"   Risk Level: {risk_level}")
    print("-" * 70)

## 💾 Save Best Models and Artifacts


In [ ]:
# Create output directory
output_dir = Path("trained_models_enhanced")
output_dir.mkdir(exist_ok=True)

# Save all enhanced models
print("💾 Saving enhanced models...\n")

joblib.dump(stacking_model, output_dir / "stacking_ensemble.pkl")
print("✅ Saved stacking_ensemble.pkl")

joblib.dump(voting_model, output_dir / "voting_ensemble.pkl")
print("✅ Saved voting_ensemble.pkl")

joblib.dump(xgb_tuned, output_dir / "xgboost_tuned.pkl")
print("✅ Saved xgboost_tuned.pkl")

joblib.dump(rf_tuned, output_dir / "random_forest_tuned.pkl")
print("✅ Saved random_forest_tuned.pkl")

joblib.dump(lgbm_model, output_dir / "lightgbm_model.pkl")
print("✅ Saved lightgbm_model.pkl")

# Save preprocessing artifacts
joblib.dump(scaler, output_dir / "scaler_enhanced.pkl")
print("✅ Saved scaler_enhanced.pkl")

joblib.dump(label_encoder, output_dir / "label_encoder_enhanced.pkl")
print("✅ Saved label_encoder_enhanced.pkl")

joblib.dump(list(X.columns), output_dir / "feature_names_enhanced.pkl")
print("✅ Saved feature_names_enhanced.pkl")

# Save model comparison
results.to_csv(output_dir / "model_comparison_enhanced.csv", index=False)
print("✅ Saved model_comparison_enhanced.csv")

# Save best hyperparameters
best_params = {
    "XGBoost": xgb_random.best_params_,
    "Random Forest": rf_random.best_params_,
}

with open(output_dir / "best_hyperparameters_enhanced.json", "w") as f:
    json.dump(best_params, f, indent=4)
print("✅ Saved best_hyperparameters_enhanced.json")

# Save engineered features list
with open(output_dir / "engineered_features.json", "w") as f:
    json.dump(engineered_features, f, indent=4)
print("✅ Saved engineered_features.json")

print(f"\n📁 All models and artifacts saved to: {output_dir.absolute()}")
print(f"\n📊 Models saved:")
print(f"   Enhanced Models: 5 models")
print(
    f"   Total Features: {X.shape[1]} (28 original + {len(engineered_features)} engineered)"
)
print(f"\n🎯 Recommended for production: {best_model_name}")
print(f"   Accuracy: {max(stacking_acc, voting_acc)*100:.2f}%")
print(
    f"   Improvement over original RF: {(max(stacking_acc, voting_acc) - 0.7514)*100:+.2f}%"
)

## 📈 Feature Importance - Enhanced Features


In [ ]:
# Get feature importance from best tree-based model
if hasattr(xgb_tuned, "feature_importances_"):
    feature_importance = pd.DataFrame(
        {"feature": X.columns, "importance": xgb_tuned.feature_importances_}
    ).sort_values("importance", ascending=False)

    print("📈 Top 20 Most Important Features (XGBoost Tuned):\n")
    print(feature_importance.head(20).to_string(index=False))

    # Visualize top 15
    plt.figure(figsize=(10, 8))
    top_15 = feature_importance.head(15)
    colors = [
        "#e74c3c" if feat in engineered_features else "#3498db"
        for feat in top_15["feature"]
    ]
    plt.barh(top_15["feature"], top_15["importance"], color=colors)
    plt.xlabel("Importance", fontsize=12)
    plt.title(
        "Top 15 Most Important Features\n(Red=Engineered, Blue=Original)",
        fontsize=14,
        fontweight="bold",
    )
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()

    # Count engineered features in top 20
    engineered_in_top20 = sum(
        1
        for feat in feature_importance.head(20)["feature"]
        if feat in engineered_features
    )
    print(f"\n✨ Engineered features in top 20: {engineered_in_top20}/20")

## 📝 Summary Report


In [ ]:
print("=" * 80)
print("🎓 ENHANCED MODEL TRAINING SUMMARY")
print("=" * 80)
print(f"\n📊 DATASET:")
print(f"   Total Samples: {len(df)}")
print(f"   Training Set: {len(X_train)} → {len(X_train_balanced)} (after SMOTE)")
print(f"   Test Set: {len(X_test)}")
print(f"\n🔧 FEATURES:")
print(f"   Original Features: 28")
print(f"   Engineered Features: {len(engineered_features)}")
print(f"   Total Features: {X.shape[1]}")
print(f"\n🏆 BEST MODEL: {best_model_name}")
print(f"   Test Accuracy: {max(stacking_acc, voting_acc)*100:.2f}%")
print(f"   Baseline RF Accuracy: 75.14%")
print(f"   Improvement: {(max(stacking_acc, voting_acc) - 0.7514)*100:+.2f}%")
print(f"\n📈 ALL MODELS TESTED:")
for idx, row in results.iterrows():
    print(f"   {row['Model']:30s} {row['Accuracy']*100:.2f}%")
print(f"\n💡 KEY IMPROVEMENTS:")
print(f"   ✅ Feature Engineering: +{len(engineered_features)} new features")
print(f"   ✅ Class Balancing: SMOTE oversampling")
print(f"   ✅ Advanced Models: XGBoost, LightGBM")
print(f"   ✅ Hyperparameter Tuning: RandomizedSearchCV (50 iterations)")
print(f"   ✅ Ensemble Methods: Stacking + Voting")
print(f"\n💾 OUTPUT:")
print(f"   Location: {output_dir.absolute()}")
print(f"   Models Saved: 5")
print(f"   Artifacts Saved: scaler, encoder, features, hyperparameters")
print("\n" + "=" * 80)